## Step 2: Risk Scoring Matrix Heuristics & User Profiling

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
BASE_DIR = Path(os.getcwd()).resolve().parent
DATA_PROCESSED = BASE_DIR / "data" / "processed"
df = pd.read_parquet(DATA_PROCESSED / "stg_master_prepared.parquet")

### Temporal & Geographic Vector Transformations

In [ ]:
df["is_weekend"] = df["day_of_week"].isin([0, 6]).astype(int)
df["is_night_transaction"] = df["hour"].between(1, 5).astype(int)
df["high_distance_flag"] = (df["distance_from_home"] > 50).astype(int)
df["is_weekend"] = df["day_of_week"].isin([0, 6]).astype(int)
df["is_night_transaction"] = df["hour"].between(1, 5).astype(int)
df["high_distance_flag"] = (df["distance_from_home"] > 50).astype(int)


df = df.sort_values(["customer_id", "transaction_time"])
df["prev_transaction_time"] = df.groupby("customer_id")["transaction_time"].shift(1)

time_delta = df["transaction_time"] - df["prev_transaction_time"]
df["minutes_since_last_txn"] = (time_delta.dt.total_seconds() / 60.0).fillna(99999)

df["rapid_txn_flag"] = (df["minutes_since_last_txn"] < 10).astype(int)

df["merchant_diversity"] = df["customer_id"].map(df.groupby("customer_id")["merchant"].nunique())
df["location_diversity"] = df["customer_id"].map(df.groupby("customer_id")["location"].nunique())

### Mathematical Matrix Categorization Layer

In [ ]:
merchant_risk_map = {
    "Cryptocurrency": 100, "Travel": 75, "Electronics": 75, "Gaming": 75,
    "Shopping Mall": 50, "Education": 50, "Healthcare": 50,
    "Restaurant": 25, "Entertainment": 25, "Groceries": 25, "Fuel": 25, "Food Delivery": 25
}

channel_risk_map = {
    "NetBanking": 100, "Card": 40, "UPI": 35
}

df["merchant_risk"] = df["merchant"].map(merchant_risk_map).fillna(25)
df["channel_risk"] = df["channel"].map(channel_risk_map).fillna(35)

df["amount_risk"] = pd.cut(
    df["amount"], 
    bins=[-float('inf'), 20000, 50000, 100000, 150000, float('inf')], 
    labels=[10, 30, 60, 80, 100]
).astype(int)

df["distance_risk"] = pd.cut(
    df["distance_from_home"], 
    bins=[-float('inf'), 5, 10, 20, 50, float('inf')], 
    labels=[10, 30, 60, 80, 100]
).astype(int)

df["time_risk"] = pd.cut(
    df["hour"], 
    bins=[-float('inf'), 5, 8, 22, float('inf')], 
    labels=[100, 50, 20, 40]
).astype(int)

### Executing Dynamic Risk Formula

In [ ]:
df["risk_score"] = (
      0.30 * df["merchant_risk"]
    + 0.25 * df["amount_risk"]
    + 0.20 * df["distance_risk"]
    + 0.15 * df["channel_risk"]
    + 0.10 * df["time_risk"]
)

df["risk_segment"] = pd.cut(
    df["risk_score"],
    bins=[-float('inf'), 25, 50, 75, float('inf')],
    labels=["Low Risk", "Medium Risk", "High Risk", "Critical Risk"]
).astype(str)

### Framework Validation Analysis

In [ ]:
print("Performance metrics across derived risk intervals:")
summary_view = df.groupby("risk_segment").agg(
    total_transactions=("transaction_id", "count"),
    avg_risk_score=("risk_score", "mean"),
    fraud_transactions=("is_fraud", "sum"),
    fraud_rate_pct=("is_fraud", "mean")
).reset_index()

print(summary_view)
customer_profiles = df[["customer_id", "risk_score", "risk_segment"]].drop_duplicates()
customer_profiles.to_csv(DATA_PROCESSED / "customer_risk_segments.csv", index=False)
df.to_parquet(DATA_PROCESSED / "stg_features_calculated.parquet", index=False)
print("[SUCCESS] Operational risk maps written out cleanly.")